# Pandas — Transformations

Notebook 03 got you to a clean `DataFrame`. Real data is rarely clean on arrival. This notebook covers the five transformations that turn raw, messy data into model-ready shape.

1. **Missing data** — finding it, dropping it, filling it.
2. **`groupby`** — split-apply-combine, the most important idiom in pandas.
3. **`merge`** — joining two DataFrames on a key.
4. **Reshape** — `pivot` (long → wide) and `melt` (wide → long).
5. **Time series** — parsing dates, indexing by date, resampling.

Master these five and most of the data cleaning you will ever do becomes mechanical.


## Missing data

Real data has gaps. Three operations cover almost everything.

- **`df.isna()`** — boolean DataFrame of the same shape, `True` where a value is missing.
- **`df.dropna()`** — drop rows (or columns) that contain any (or all) missing values.
- **`df.fillna(value)`** — fill missing values with a constant, a per-column dict, or a method like forward-fill.

Pandas represents missing as `NaN` for floats and `pd.NA` for the newer nullable dtypes. The line you should run **immediately** after loading any DataFrame is `df.isna().sum()` — a count of nulls per column. That single result tells you which columns need attention before anything else.


In [ ]:
import numpy as np
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "name": ["Aarav Sharma", "Diya Patel", "Vihaan Reddy", "Ananya Iyer", None, "Arjun Nair"],
    "age": [27, 34, np.nan, 29, 52, 31],
    "city": ["Bengaluru", "Mumbai", "Hyderabad", None, "Bengaluru", "Pune"],
    "monthly_income": [45000, 82000, 120000, 67000, np.nan, 58000],
})

print("nulls per column:")
print(customers.isna().sum())

print("\ndropna() — keep only rows with no missing values:")
print(customers.dropna())

# Per-column fill — different default per column
filled = customers.fillna({
    "name": "(unknown)",
    "age": customers["age"].median(),
    "city": "(unknown)",
    "monthly_income": customers["monthly_income"].median(),
})
print("\nfillna with per-column defaults:")
print(filled)


## `groupby` — split, apply, combine

This is the most important idiom in pandas. Anything you would write in SQL as `GROUP BY`, you write in pandas as `groupby`.

The three steps are always the same.

1. **Split** the DataFrame into groups on one or more columns.
2. **Apply** a function to each group — usually an aggregation like `mean`, `sum`, `count`.
3. **Combine** the per-group results back into a single DataFrame or Series.

Pandas does all three in one chained call: `df.groupby("city")["monthly_income"].mean()`. Read it left to right — group by city, pick the income column, take the mean per group.


In [ ]:
customers = pd.DataFrame({
    "customer_id": range(1001, 1011),
    "city": ["Bengaluru", "Mumbai", "Bengaluru", "Pune", "Mumbai",
             "Bengaluru", "Pune", "Mumbai", "Bengaluru", "Pune"],
    "segment": ["retail", "retail", "wealth", "retail", "wealth",
                "retail", "wealth", "retail", "wealth", "retail"],
    "age": [27, 34, 45, 29, 52, 31, 38, 26, 41, 33],
    "monthly_income": [45000, 82000, 220000, 67000, 195000,
                       58000, 175000, 71000, 240000, 64000],
})

# Single column, single aggregation
print(customers.groupby("city")["monthly_income"].mean().round(0))


You can group by multiple columns and apply multiple aggregations at once with `.agg(...)`. The result is indexed by the group keys.

The cleanest form uses **named aggregations** — `column_name = ("source_col", "agg_func")`. It produces neat output column names without the multi-level mess you get from passing a dict.


In [ ]:
result = (
    customers
    .groupby(["city", "segment"])
    .agg(
        customers=("customer_id", "count"),
        avg_income=("monthly_income", "mean"),
        max_age=("age", "max"),
    )
    .round(0)
)
print(result)


## `merge` — joining two DataFrames

`merge` is the pandas equivalent of a SQL JOIN. The `how` argument controls which side you keep.

| `how` | Keeps |
|---|---|
| `"inner"` (default) | Only rows where the key matches in both tables |
| `"left"` | All rows from the left table; `NaN` where the right has no match |
| `"right"` | All rows from the right table; `NaN` where the left has no match |
| `"outer"` | All rows from both; `NaN` where either side has no match |

Specify the join column with `on="key"`. If the two tables call the same logical key by different names, use `left_on=` and `right_on=`.


In [ ]:
customers = pd.DataFrame({
    "customer_id": [1001, 1002, 1003, 1004, 1005],
    "name": ["Aarav", "Diya", "Vihaan", "Ananya", "Arjun"],
    "city": ["Bengaluru", "Mumbai", "Hyderabad", "Chennai", "Bengaluru"],
})

loans = pd.DataFrame({
    "loan_id": [5001, 5002, 5003, 5004],
    "customer_id": [1001, 1001, 1003, 1006],   # 1006 doesn't exist in customers
    "product": ["home_loan", "personal_loan", "car_loan", "home_loan"],
    "principal": [2_500_000, 200_000, 800_000, 3_000_000],
})

inner = customers.merge(loans, on="customer_id", how="inner")
print("INNER — only customers with loans, only loans whose customer exists:")
print(inner)


In [ ]:
left = customers.merge(loans, on="customer_id", how="left")
print("LEFT — every customer kept; loan columns are NaN if no match:")
print(left)

# Counting unmatched rows is a one-liner
print("\ncustomers with no loans:", left["loan_id"].isna().sum())


## Reshape — `pivot` and `melt`

The same data can live in two shapes.

- **Long form** — one row per observation, columns are usually `(id, attribute, value)`. Great for storage, event logs, time series.
- **Wide form** — one row per entity, many attribute columns. Great for spreadsheets, correlation matrices, and most plotting libraries.

Two functions move between them.

- **`df.pivot(index, columns, values)`** — long → wide. Each unique value of `columns` becomes a new column.
- **`pd.melt(df, id_vars, value_vars)`** — wide → long. Many columns collapse into one `variable` + `value` pair.

You will reshape data more than you expect. Time-series aggregates land long; `seaborn` heatmaps and correlation matrices want wide; categorical plots often want long again.


In [ ]:
# Long form: one row per (customer, product) pair
loans_long = pd.DataFrame({
    "customer_id": [1001, 1001, 1002, 1003, 1003, 1004],
    "product":     ["home_loan", "car_loan", "home_loan", "car_loan", "personal_loan", "home_loan"],
    "principal":   [2_500_000, 800_000, 1_800_000, 600_000, 200_000, 3_200_000],
})
print("LONG form:")
print(loans_long)

# Wide form: one row per customer, one column per product
loans_wide = loans_long.pivot(index="customer_id", columns="product", values="principal")
print("\nWIDE form (pivot):")
print(loans_wide)


In [ ]:
# Reverse direction: wide → long with melt
back_to_long = (
    loans_wide
    .reset_index()
    .melt(
        id_vars="customer_id",
        value_vars=["home_loan", "car_loan", "personal_loan"],
        var_name="product",
        value_name="principal",
    )
    .dropna()
    .sort_values(["customer_id", "product"])
    .reset_index(drop=True)
)
print("LONG form (melt):")
print(back_to_long)


## Time series — parsing, indexing, resampling

Two ingredients turn ordinary columns into a time series.

1. Convert the date column to a real `datetime64` dtype with `pd.to_datetime(...)`.
2. Set that column as the DataFrame's index via `set_index(...)`.

Once the index is `datetime`, slicing by date range just works — `df.loc["2026-02"]` returns every row in February 2026. And you unlock **`resample`** — `groupby` for time, with calendar-aware buckets instead of arbitrary keys.


In [ ]:
repayments = pd.DataFrame({
    "date": [
        "2026-01-05", "2026-01-15", "2026-02-05", "2026-02-15",
        "2026-02-28", "2026-03-05", "2026-03-15", "2026-03-31",
    ],
    "loan_id": [5001, 5002, 5001, 5002, 5003, 5001, 5002, 5003],
    "amount":  [25000, 12000, 25000, 12000, 8000, 25000, 12000, 8000],
})

# Parse the date column — one of the most important lines you will write
repayments["date"] = pd.to_datetime(repayments["date"])
repayments = repayments.set_index("date").sort_index()

print("dtypes:")
print(repayments.dtypes)
print("\nslicing — all of February 2026:")
print(repayments.loc["2026-02"])


`resample` is `groupby` for time. You bucket rows into calendar windows — `"D"` for day, `"W"` for week, `"ME"` for month-end, `"QE"` for quarter-end — then aggregate just like a normal `groupby`.

The codes are short and memorable; the full list lives in the pandas docs under "offset aliases."


In [ ]:
# Monthly total repayments across all loans
monthly = repayments["amount"].resample("ME").sum()
print("monthly totals:")
print(monthly)

# Monthly total per loan — groupby + resample + unstack into wide form
per_loan_monthly = (
    repayments
    .groupby("loan_id")["amount"]
    .resample("ME")
    .sum()
    .unstack(0)
    .fillna(0)
    .astype(int)
)
print("\nmonthly totals per loan (wide form):")
print(per_loan_monthly)


## What's next

You now have the five transformations that cover the bulk of real-world data cleaning: missing-data handling, `groupby`, `merge`, `pivot`/`melt`, and time-series resampling.

Notebook 05 introduces **visualization** — `matplotlib`'s figure/axes model and `seaborn`'s statistical plots. The cleaned, reshaped DataFrames you build here are exactly what gets fed into plots in the next notebook.

Two from-memory exercises.

1. Build a `card_transactions` DataFrame with `customer_id`, `category`, `amount`, and `timestamp`. Insert a few `NaN`s in `category`. Fill them with the string `"unknown"`, then `groupby` category and compute the total spend per category.
2. Parse the `timestamp`, set it as the index, and resample those transactions by week (`"W"`) to get a weekly total spend Series.
